In [1]:
# input
site_anno_file="../mbp_site_anno/data/entryId-seqNum-resi-metalResi.tsv"
pfam_region_file="../../_database/Pfam/Pfam-A.regions.tsv"
# output
mbp_pfam_file="data/entryId-pfamId.tsv"

In [2]:
import pandas as pd

df_pfam_region = pd.read_table(pfam_region_file, usecols=["pfamseq_acc", "pfamA_acc", "seq_start", "seq_end"])
df_mbp_site_anno = pd.read_table(site_anno_file, header=None, names=["pfamseq_acc", "seq_nums", "_", "__"], usecols=["pfamseq_acc", "seq_nums"])

In [3]:
import tqdm


id2nums = dict(zip(df_mbp_site_anno["pfamseq_acc"], df_mbp_site_anno["seq_nums"]))
df_pfam_region = df_pfam_region[df_pfam_region["pfamseq_acc"].map(lambda x: x in id2nums.keys())]

records = []
for (seq_id, ), df_seq in tqdm.tqdm(df_pfam_region.groupby(by=["pfamseq_acc"])):
    
    nums = [int(i) for i in id2nums[seq_id].split(",")]
    pfams = []
    for _, row in df_seq.iterrows():
        pfam = row['pfamA_acc']
        start = row['seq_start']
        end = row['seq_end']

        has_inter = any([i >= start and i <= end for i in nums])
        if has_inter:
            pfams.append(pfam)
    
    if len(pfams) != 0:
        records.append({
            "seq_id": seq_id,
            "pfam_id": ",".join(pfams)
        })

100%|██████████| 2086113/2086113 [04:22<00:00, 7961.37it/s]


In [4]:
pd.DataFrame(records).to_csv(mbp_pfam_file, sep="\t", header=None, index=None)